In [ ]:
# ============================================================
# ASR ERROR ANALYSIS TOOLKIT
# Comprehensive analysis of noise types in ASR corrections
# ============================================================

import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import SequenceMatcher

# ============================================================
# PERSIAN PHONETIC GROUPS (for homophone analysis)
# ============================================================
PHONETIC_GROUPS = {
    'z': ['ز', 'ذ', 'ض', 'ظ'],       # /z/ sound
    's': ['س', 'ص', 'ث'],            # /s/ sound
    't': ['ت', 'ط'],                 # /t/ sound
    'h': ['ه', 'ح'],                 # /h/ sound
    'gh': ['ق', 'غ'],                # /gh/ sound
    'a': ['ا', 'آ', 'أ', 'إ', 'ٱ'],  # /a/ sound
    'i': ['ی', 'ئ', 'ي'],            # /i/ sound
    'v': ['و', 'ؤ'],                 # /v/ sound
    'eh': ['ۀ', 'ة'],                # /eh/
}

# Build reverse mapping: character -> phonetic group
CHAR_TO_GROUP = {}
for group_name, chars in PHONETIC_GROUPS.items():
    for ch in chars:
        CHAR_TO_GROUP[ch] = group_name


# ============================================================
# UTILITY FUNCTIONS
# ============================================================
def levenshtein_distance(s1, s2):
    """Compute Levenshtein distance between two strings."""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]


def is_homophone_pair(c1, c2):
    """Check if two characters belong to the same phonetic group."""
    g1 = CHAR_TO_GROUP.get(c1)
    g2 = CHAR_TO_GROUP.get(c2)
    return g1 is not None and g1 == g2


def align_characters(s1, s2):
    """Align two strings using SequenceMatcher to find edit operations."""
    sm = SequenceMatcher(None, s1, s2)
    operations = []
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        operations.append({
            'tag': tag,
            'ref_start': i1, 'ref_end': i2, 'ref_chars': s1[i1:i2],
            'hyp_start': j1, 'hyp_end': j2, 'hyp_chars': s2[j1:j2]
        })
    return operations


def classify_character_error(ref_char, hyp_char):
    """Classify a character-level substitution error."""
    if ref_char == hyp_char:
        return 'match'
    if is_homophone_pair(ref_char, hyp_char):
        return 'homophone_substitution'
    return 'non_homophone_substitution'


# ============================================================
# ERROR CLASSIFICATION AT WORD LEVEL
# ============================================================
def classify_word_error(ref_word, hyp_word):
    """
    Classify a word-level error:
    - 'exact': identical
    - 'homophone': same pronunciation but different spelling
    - 'near_miss': 1-2 character difference
    - 'semantic_swap': completely different word (both ASR wrong)
    - 'merge/split': word boundary issues
    """
    if ref_word == hyp_word:
        return 'exact'
    
    # Character-level alignment for this word
    ops = align_characters(ref_word, hyp_word)
    
    # Count operations
    n_sub = 0
    n_del = 0
    n_ins = 0
    n_homophone_subs = 0
    
    for op in ops:
        if op['tag'] == 'replace':
            min_len = min(len(op['ref_chars']), len(op['hyp_chars']))
            for i in range(min_len):
                if is_homophone_pair(op['ref_chars'][i], op['hyp_chars'][i]):
                    n_homophone_subs += 1
                else:
                    n_sub += 1
            n_sub += abs(len(op['ref_chars']) - len(op['hyp_chars']))
        elif op['tag'] == 'delete':
            n_del += len(op['ref_chars'])
        elif op['tag'] == 'insert':
            n_ins += len(op['hyp_chars'])
    
    total_edits = n_sub + n_del + n_ins
    dist = levenshtein_distance(ref_word, hyp_word)
    
    # If all substitutions are homophones
    if n_sub == 0 and n_del == 0 and n_ins == 0 and n_homophone_subs > 0:
        return 'homophone'
    
    # Near miss: small edit distance
    if dist <= 2:
        if n_homophone_subs > n_sub:
            return 'homophone'
        return 'near_miss'
    
    # Otherwise, unrelated word (semantic swap)
    return 'semantic_swap'


# ============================================================
# MAIN ANALYSIS FUNCTION
# ============================================================
def analyze_errors(csv_path):
    """Run complete error analysis on the given CSV file."""
    
    df = pd.read_csv(csv_path)
    print("=" * 80)
    print("ASR ERROR ANALYSIS REPORT")
    print("=" * 80)
    print(f"Total samples: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print()
    
    # ============================================================
    # 1. OVERALL STATISTICS
    # ============================================================
    print("=" * 80)
    print("1. OVERALL STATISTICS")
    print("=" * 80)
    
    n_correct = df['is_correct'].sum()
    n_total = len(df)
    print(f"Correct predictions:   {n_correct} ({100*n_correct/n_total:.2f}%)")
    print(f"Incorrect predictions: {n_total - n_correct} ({100*(n_total-n_correct)/n_total:.2f}%)")
    print(f"Mean CER: {df['cer'].mean():.4f}")
    print(f"Median CER: {df['cer'].median():.4f}")
    print(f"Mean WER: {df['wer'].mean():.4f}")
    print(f"Median WER: {df['wer'].median():.4f}")
    print()
    
    # ============================================================
    # 2. BASELINE COMPARISON
    # ============================================================
    print("=" * 80)
    print("2. BASELINE COMPARISON (Whisper vs MMS vs Model)")
    print("=" * 80)
    
    def compute_simple_cer(ref, hyp):
        if len(ref) == 0:
            return 1.0 if len(hyp) > 0 else 0.0
        return levenshtein_distance(ref, hyp) / len(ref)
    
    whisper_cers = []
    mms_cers = []
    model_cers = []
    
    for _, row in df.iterrows():
        gt = str(row['ground_truth'])
        whisper_cers.append(compute_simple_cer(gt, str(row['whisper_input'])))
        mms_cers.append(compute_simple_cer(gt, str(row['mms_input'])))
        model_cers.append(compute_simple_cer(gt, str(row['model_prediction'])))
    
    print(f"Whisper  - Mean CER: {np.mean(whisper_cers):.4f}")
    print(f"MMS      - Mean CER: {np.mean(mms_cers):.4f}")
    print(f"Model    - Mean CER: {np.mean(model_cers):.4f}")
    print()
    
    # ============================================================
    # 3. CHARACTER-LEVEL EDIT OPERATIONS (per sample)
    # ============================================================
    print("=" * 80)
    print("3. CHARACTER-LEVEL ERROR BREAKDOWN (Model vs Ground Truth)")
    print("=" * 80)
    
    total_subs = 0
    total_dels = 0
    total_ins = 0
    homophone_subs = 0
    non_homophone_subs = 0
    
    for _, row in df.iterrows():
        ref = str(row['ground_truth'])
        hyp = str(row['model_prediction'])
        ops = align_characters(ref, hyp)
        
        for op in ops:
            if op['tag'] == 'replace':
                min_len = min(len(op['ref_chars']), len(op['hyp_chars']))
                for i in range(min_len):
                    total_subs += 1
                    if is_homophone_pair(op['ref_chars'][i], op['hyp_chars'][i]):
                        homophone_subs += 1
                    else:
                        non_homophone_subs += 1
                total_subs += abs(len(op['ref_chars']) - len(op['hyp_chars']))
            elif op['tag'] == 'delete':
                total_dels += len(op['ref_chars'])
            elif op['tag'] == 'insert':
                total_ins += len(op['hyp_chars'])
    
    total_errors = total_subs + total_dels + total_ins
    if total_errors > 0:
        print(f"Total character-level errors: {total_errors}")
        print(f"  Substitutions: {total_subs} ({100*total_subs/total_errors:.2f}%)")
        print(f"    - Homophone subs:      {homophone_subs} ({100*homophone_subs/max(total_subs,1):.2f}% of subs)")
        print(f"    - Non-homophone subs:  {non_homophone_subs} ({100*non_homophone_subs/max(total_subs,1):.2f}% of subs)")
        print(f"  Deletions:     {total_dels} ({100*total_dels/total_errors:.2f}%)")
        print(f"  Insertions:    {total_ins} ({100*total_ins/total_errors:.2f}%)")
    print()
    
    # ============================================================
    # 4. WORD-LEVEL ERROR CLASSIFICATION
    # ============================================================
    print("=" * 80)
    print("4. WORD-LEVEL ERROR CLASSIFICATION")
    print("=" * 80)
    
    word_error_types = Counter()
    word_error_examples = defaultdict(list)
    
    for _, row in df.iterrows():
        ref_words = str(row['ground_truth']).split()
        hyp_words = str(row['model_prediction']).split()
        
        sm = SequenceMatcher(None, ref_words, hyp_words)
        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            if tag == 'replace':
                for k in range(max(i2 - i1, j2 - j1)):
                    ref_w = ref_words[i1 + k] if i1 + k < i2 else ''
                    hyp_w = hyp_words[j1 + k] if j1 + k < j2 else ''
                    if ref_w and hyp_w:
                        etype = classify_word_error(ref_w, hyp_w)
                        word_error_types[etype] += 1
                        if len(word_error_examples[etype]) < 5:
                            word_error_examples[etype].append((ref_w, hyp_w))
                    elif ref_w:
                        word_error_types['deletion'] += 1
                    elif hyp_w:
                        word_error_types['insertion'] += 1
            elif tag == 'delete':
                for k in range(i1, i2):
                    word_error_types['deletion'] += 1
            elif tag == 'insert':
                for k in range(j1, j2):
                    word_error_types['insertion'] += 1
    
    total_word_errors = sum(word_error_types.values())
    print(f"Total word-level errors: {total_word_errors}")
    for etype, count in word_error_types.most_common():
        pct = 100 * count / max(total_word_errors, 1)
        print(f"  {etype:25s}: {count:5d} ({pct:5.2f}%)")
        if word_error_examples[etype]:
            print(f"      Examples: {word_error_examples[etype][:3]}")
    print()
    
    # ============================================================
    # 5. CHARACTER SUBSTITUTION MATRIX (Most frequent confusions)
    # ============================================================
    print("=" * 80)
    print("5. MOST FREQUENT CHARACTER CONFUSIONS")
    print("=" * 80)
    
    confusion_counter = Counter()
    for _, row in df.iterrows():
        ref = str(row['ground_truth'])
        hyp = str(row['model_prediction'])
        ops = align_characters(ref, hyp)
        for op in ops:
            if op['tag'] == 'replace':
                min_len = min(len(op['ref_chars']), len(op['hyp_chars']))
                for i in range(min_len):
                    if op['ref_chars'][i] != op['hyp_chars'][i]:
                        confusion_counter[(op['ref_chars'][i], op['hyp_chars'][i])] += 1
    
    print("Top 20 character confusions (correct -> predicted):")
    for (ref_c, hyp_c), count in confusion_counter.most_common(20):
        is_homo = "HOMOPHONE" if is_homophone_pair(ref_c, hyp_c) else "non-homo"
        print(f"  '{ref_c}' -> '{hyp_c}': {count:4d} times  [{is_homo}]")
    print()
    
    # ============================================================
    # 6. NOISE TYPE DISTRIBUTION
    # ============================================================
    print("=" * 80)
    print("6. NOISE TYPE DISTRIBUTION (Summary)")
    print("=" * 80)
    
    homophone_ratio = homophone_subs / max(total_subs, 1)
    non_homophone_ratio = non_homophone_subs / max(total_subs, 1)
    
    if homophone_ratio > 0.5:
        noise_type = "MOSTLY HOMOPHONE-BASED (systematic phonetic bias)"
    elif homophone_ratio > 0.3:
        noise_type = "MIXED (semi-systematic: homophones + random errors)"
    else:
        noise_type = "MOSTLY RANDOM (non-homophone errors dominate)"
    
    print(f"Noise type classification: {noise_type}")
    print(f"  Homophone substitution ratio:     {homophone_ratio:.3f}")
    print(f"  Non-homophone substitution ratio: {non_homophone_ratio:.3f}")
    print()
    
    # ============================================================
    # 7. ERROR VS SENTENCE LENGTH
    # ============================================================
    print("=" * 80)
    print("7. ERROR VS SENTENCE LENGTH")
    print("=" * 80)
    
    df['gt_len'] = df['ground_truth'].apply(lambda x: len(str(x)))
    df['len_bucket'] = pd.cut(df['gt_len'], bins=[0, 15, 30, 50, 100, 500],
                              labels=['very_short', 'short', 'medium', 'long', 'very_long'])
    
    length_analysis = df.groupby('len_bucket').agg({
        'cer': 'mean',
        'wer': 'mean',
        'is_correct': 'sum'
    }).round(4)
    length_analysis['count'] = df.groupby('len_bucket').size()
    print(length_analysis)
    print()
    
    # ============================================================
    # 8. IDENTIFY PROBLEMATIC PATTERNS
    # ============================================================
    print("=" * 80)
    print("8. PROBLEMATIC PATTERNS")
    print("=" * 80)
    
    # 8a. Repetition in model output
    print("\n[8a] Repetition analysis in model output:")
    repetition_count = 0
    repetition_examples = []
    for _, row in df.iterrows():
        pred = str(row['model_prediction'])
        words = pred.split()
        for i in range(len(words) - 1):
            if words[i] == words[i+1] and len(words[i]) > 1:
                repetition_count += 1
                if len(repetition_examples) < 5:
                    repetition_examples.append(pred)
                break
    
    print(f"  Samples with consecutive word repetition: {repetition_count}")
    for ex in repetition_examples:
        print(f"    - {ex[:80]}")
    
    # 8b. High-CER samples
    print("\n[8b] Samples with very high CER (>0.5):")
    high_cer = df[df['cer'] > 0.5]
    print(f"  Count: {len(high_cer)} ({100*len(high_cer)/len(df):.2f}%)")
    for _, row in high_cer.head(5).iterrows():
        print(f"    GT:    {row['ground_truth'][:70]}")
        print(f"    Whisper: {row['whisper_input'][:70]}")
        print(f"    MMS:     {row['mms_input'][:70]}")
        print(f"    Model:   {row['model_prediction'][:70]}")
        print(f"    CER: {row['cer']:.3f}")
        print()
    
    # 8c. Cases where both ASRs wrong but model fixed it
    print("\n[8c] Cases where model CORRECTED both ASR errors:")
    fixed_count = 0
    for _, row in df.iterrows():
        gt = str(row['ground_truth'])
        whisper_correct = (str(row['whisper_input']) == gt)
        mms_correct = (str(row['mms_input']) == gt)
        model_correct = (str(row['model_prediction']) == gt)
        if not whisper_correct and not mms_correct and model_correct:
            fixed_count += 1
    print(f"  Cases where both ASRs wrong but model correct: {fixed_count}")
    
    # 8d. Cases where both ASRs correct but model wrong (hallucination)
    print("\n[8d] Cases where model INTRODUCED errors (both ASRs were correct):")
    hallucination_count = 0
    hallucination_examples = []
    for _, row in df.iterrows():
        gt = str(row['ground_truth'])
        whisper_correct = (str(row['whisper_input']) == gt)
        mms_correct = (str(row['mms_input']) == gt)
        model_correct = (str(row['model_prediction']) == gt)
        if whisper_correct and mms_correct and not model_correct:
            hallucination_count += 1
            if len(hallucination_examples) < 5:
                hallucination_examples.append((gt, str(row['model_prediction'])))
    print(f"  Model hallucinations (both ASRs correct, model wrong): {hallucination_count}")
    for gt, pred in hallucination_examples:
        print(f"    GT:    {gt}")
        print(f"    Model: {pred}")
        print()
    
    # ============================================================
    # 9. VISUALIZATION
    # ============================================================
    print("=" * 80)
    print("9. GENERATING VISUALIZATIONS...")
    print("=" * 80)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: CER distribution
    axes[0, 0].hist(df['cer'], bins=50, color='steelblue', edgecolor='black')
    axes[0, 0].axvline(df['cer'].mean(), color='red', linestyle='--', label=f"Mean: {df['cer'].mean():.3f}")
    axes[0, 0].set_xlabel('CER')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('CER Distribution')
    axes[0, 0].legend()
    
    # Plot 2: Word error types
    if word_error_types:
        types = list(word_error_types.keys())
        counts = [word_error_types[t] for t in types]
        axes[0, 1].barh(types, counts, color='coral')
        axes[0, 1].set_xlabel('Count')
        axes[0, 1].set_title('Word Error Type Distribution')
    
    # Plot 3: Character edit operations
    edit_ops = ['Substitutions\n(homophone)', 'Substitutions\n(non-homo)', 'Deletions', 'Insertions']
    edit_counts = [homophone_subs, non_homophone_subs, total_dels, total_ins]
    colors = ['green', 'orange', 'red', 'purple']
    axes[1, 0].bar(edit_ops, edit_counts, color=colors)
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Character-Level Edit Operations')
    axes[1, 0].tick_params(axis='x', rotation=15)
    
    # Plot 4: CER vs sentence length
    axes[1, 1].scatter(df['gt_len'], df['cer'], alpha=0.3, s=10, color='teal')
    axes[1, 1].set_xlabel('Ground Truth Length (chars)')
    axes[1, 1].set_ylabel('CER')
    axes[1, 1].set_title('CER vs Sentence Length')
    
    plt.tight_layout()
    plt.savefig('./error_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("Visualization saved to ./error_analysis.png")
    print()
    
    # ============================================================
    # 10. FINAL SUMMARY
    # ============================================================
    print("=" * 80)
    print("10. FINAL SUMMARY")
    print("=" * 80)
    print(f"Total samples analyzed: {len(df)}")
    print(f"Model correct: {n_correct}/{n_total} ({100*n_correct/n_total:.1f}%)")
    print(f"Homophone substitution ratio: {homophone_ratio:.3f}")
    print(f"Most common error type: {word_error_types.most_common(1)[0][0] if word_error_types else 'N/A'}")
    print(f"Noise characterization: {noise_type}")
    print()
    print("Interpretation:")
    if homophone_ratio > 0.5:
        print("  → The noise is SYSTEMATIC: homophone-based errors dominate.")
        print("    This means the model struggles with Persian homophone selection.")
    elif homophone_ratio > 0.3:
        print("  → The noise is SEMI-SYSTEMATIC: mixed homophone + random errors.")
        print("    Both orthographic patterns and semantic context matter.")
    else:
        print("  → The noise is MOSTLY RANDOM: non-homophone errors dominate.")
        print("    Model needs better contextual understanding of full sentences.")
    print("=" * 80)
    
    return df


# ============================================================
# RUN THE ANALYSIS
# ============================================================
if __name__ == "__main__":
    csv_file = "./results/character model results/test results.csv"  # Change path if needed
    results_df = analyze_errors(csv_file)